# Admixture proportions for a single individual (fastNGSadmix)

**Purpose.** Estimate the ancestry of **one** individual against a fixed reference panel.
NGSadmix needs many individuals to find the ancestral populations; fastNGSadmix instead
takes the populations as given and asks only where a single sample sits among them —
which is what you do with an ancient genome, a forensic sample, or one new individual.

**What you will do**
 - look at how a reference panel is stored: allele frequencies plus the number of
   individuals behind each one
 - run fastNGSadmix on a single low depth sample
 - repeat for samples with different ancestry and different amounts of data
 - see what happens when the true source population is missing from the panel

**The data.** A reference panel of **7 populations**, 195 individuals in total:

| Population | n |
|---|---|
| French | 25 |
| Han | 33 |
| Chukchi | 23 |
| Karitiana | 12 |
| Papuan | 14 |
| Sindhi | 18 |
| Yoruba | 70 |

and three single samples (`sample1`, `sample2`, `sample3`) as genotype likelihoods, to be
placed against that panel.

**Before this** do [Admixture proportions from low depth sequencing](admixture_low_depth_human.ipynb).

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data or the software moves, this is the ONLY cell you
# need to change. No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA=/course/data/current_data/admixture/human_refpanel

# the input files are linked into the working folder, so that is where they are read from
inputpath=.

# the program (on PATH here)
fastNGSadmix=fastNGSadmix

# where you will do the exercise
WORK_DIR=$HOME/admixture_reference_panel_human

mkdir -p $WORK_DIR
cd $WORK_DIR

# the cells below read the paths back out of this file, so they are only set here
cat > $WORK_DIR/env.sh <<EOF
export DATA=$DATA
export inputpath=$inputpath
export fastNGSadmix=$fastNGSadmix
export WORK=$WORK_DIR
EOF

# link the input files into the working folder
cp -sf $DATA/* . 2>/dev/null

echo --programs that are installed:--
which $fastNGSadmix

echo; echo --- files in folder ---
ls

In [ ]:
# R cannot source env.sh, so read the paths out of it rather than repeating them
env <- readLines(path.expand("~/admixture_reference_panel_human/env.sh"))
getvar <- function(k) sub(paste0('^export ', k, '='), '', grep(paste0('^export ', k, '='), env, value = TRUE)[1])
DATA <- getvar("DATA"); WORK <- getvar("WORK")
setwd(WORK)
getwd()


In [ ]:
source ~/admixture_reference_panel_human/env.sh
cd $WORK   # paths + cd to the working folder

# Set path for all input files you will use in this exercise
inputpath=


## Explore the files with the reference panel
As reference panel we will use data from these 7 populations:

| Population code/name | Description                                    | 
|-----------------|------------------------------------------------|
| French |	French individuals |
| Han	 |  Chinese individuals|
| Chukchi|	Siberian individuals |
| Karitiana	| Native American individuals |
| Papuan |	Individuals from Papua New Guinea, Melanesia |
| Sindhi |	Individuals from India |
| YRI	 | Yoruba individuals from Nigeria |

The files with genotype likelihood (GL) data from sample1, sample2 and sample3 are in beagle format — so exactly the same format as the input files you used for NGSadmix. So let's not spend time on looking at those. But before we start analysing the data then have a quick look at the files with the reference panel (nInd.txt and refPanel.txt) so you know how they look (in case you at some point want to create your own reference panel - which there are scripts for that comes with fastNGSadmix). You can do this by running the following commands:

In [ ]:
source ~/admixture_reference_panel_human/env.sh
cd $WORK   # paths + cd to the working folder

# Show the full content of the file nInd.txt
# (a file that has info how many samples from each population the panel consists of)
echo "Content of nInd.txt:"
cat ${inputpath}/nInd.txt | column -t

# Show the top 2 lines of the file refPanel.txt
# (a file that has info about allele frequencies for the 7 populations)
echo ""
echo "First 2 lines of refPanel.txt"
head -n2 ${inputpath}/refPanel.txt | column -t


- How many samples from each population does the reference population consist of?
- What are the allele frequencies of the first SNP for each of the populations?


## Analyse the samples with fastNGSadmix

Let's try to run fastNGSadmix on the 3 samples one at a time with the following commands:

In [ ]:
source ~/admixture_reference_panel_human/env.sh
cd $WORK   # paths + cd to the working folder

# Analyse sample1
$fastNGSadmix -likes ${inputpath}/sample1.beagle -fname ${inputpath}/refPanel.txt -Nname ${inputpath}/nInd.txt -outfiles sample1 -whichPops all -conv 10 -seed 1

# Analyse sample2
$fastNGSadmix -likes ${inputpath}/sample2.beagle.gz -fname ${inputpath}/refPanel.txt -Nname ${inputpath}/nInd.txt -outfiles sample2 -whichPops all -conv 10 -seed 1

# Analyse sample3
$fastNGSadmix -likes ${inputpath}/sample3.beagle.gz -fname ${inputpath}/refPanel.txt -Nname ${inputpath}/nInd.txt -outfiles sample3 -whichPops all -conv 10 -seed 1


**Questions**
 - The panel stores an allele frequency per population plus the number of individuals it was estimated from. Why does the sample size matter as well as the frequency?
 - What would go wrong if a population were represented by only two individuals?

As you can see the way to run it is similar to NGSadmix. The options -likes and -outfiles are the same (-outfiles is the equivalent of -o in NGSadmix). But now we also have the -fname and -Nname, which allows you to specify files with your reference panel. Also notice you can ask to run multiple runs with different starting points using the option -conv which makes it easier to ensure convergence. And then there is actually one more parameter that has to be set, namely -whichPops which allows you to specify that you only want to use a subset of the populations in the reference panel, or that you want to analyze all populations. So e.g. try to re-analyse sample1 using only 6 of the 7 populations in your reference panel (excluding the French) by running the same code as before except with -whichPops changed:

In [ ]:
source ~/admixture_reference_panel_human/env.sh
cd $WORK   # paths + cd to the working folder

# Re-analyse sample1 with a smaller reference panel
$fastNGSadmix -likes ${inputpath}/sample1.beagle -fname ${inputpath}/refPanel.txt -Nname ${inputpath}/nInd.txt -outfiles sample1V2 -whichPops Han,Yoruba,Sindhi,Papuan,Chukchi,Karitiana -conv 10 -seed 1


## Take a look at the output files

The output is very similar to that of NGSadmix. There is no fopt file, but there is a log file and and qopt file.

Try to look in the log files for the four analyses using the command cat, so e.g. for sample1 type:

In [ ]:
source ~/admixture_reference_panel_human/env.sh
cd $WORK   # paths + cd to the working folder

# Show content of sample1.log 
cat sample1.log


- How many loci are the 4 different analyses based on (this is in the log files and is called Overlap)?

Next try to have a look at the qopt file for sample 1 (which like for NGSadmix contains the estimated admixture proportion for the sample):

In [ ]:
source ~/admixture_reference_panel_human/env.sh
cd $WORK   # paths + cd to the working folder

# Show content of sample1.qopt
cat sample1.qopt | column -t


- Does the sample look admixed?

## Plot the analysis results

Instead of reading all the qopt files one at a time, let us plot the results of all 4 analyses in R:

In [ ]:

# Plot results for first analysis of sample1
admix<-read.table("sample1.qopt",as.is=T,h=T)
barplot(as.matrix(admix),ylab="Admixture proportion",col="red",ylim=c(0,1),main="Sample1")

# Plot results for 2nd analysis of sample1 (the one where we excluded French from the reference panel)
admix<-read.table("sample1V2.qopt",as.is=T,h=T)
barplot(as.matrix(admix),ylab="Admixture proportion",col="red",ylim=c(0,1),main="Sample1 (re-analysed)")

# Plot results for analysis of sample2
admix<-read.table("sample2.qopt",as.is=T,h=T)
barplot(as.matrix(admix),ylab="Admixture proportion",col="red",ylim=c(0,1),main="Sample2")

# Plot results for analysis of sample3
admix<-read.table("sample3.qopt",as.is=T,h=T)
barplot(as.matrix(admix),ylab="Admixture proportion",col="red",ylim=c(0,1),main="Sample3")


Based on the results: 
- what ancestry do you think the three samples have (ignore the second analysis of sample1 for now)?

Now look at the results of the second analysis of sample1 (for which a different reference panel was used). 

- Why do you think the result depends on the reference panel and what are the consequences?

Finally

- Do you trust the results for sample2 and sample3 given the number of loci it is based on?

In order to investigate this we can let fastNGSadmix run with bootstraps, where we randomly sample (with replacement), the sites the analysis is based on. This tells us something about how susceptible our estimates are to change. Try to run fastNGSadmix with 100 bootstraps for sample2 and sample3:

### How sure are we?

A point estimate on its own says nothing about how much data it rests on. sample2 and sample3 differ
a lot in how many sites they share with the reference panel, and an estimate based on a few hundred
sites can move a long way if you happen to look at a different few hundred sites.

The `-boot` option measures exactly that: it resamples the sites with replacement 100 times and
re-estimates the proportions from each resampled set. The spread of those 100 estimates is a
confidence interval for the estimate. Note what it does *not* cover - resampling sites cannot tell
you that a source population is missing from your panel, so narrow intervals are not the same as a
correct answer.

In [ ]:
source ~/admixture_reference_panel_human/env.sh
cd $WORK   # paths + cd to the working folder

$fastNGSadmix -likes ${inputpath}/sample2.beagle.gz -fname ${inputpath}/refPanel.txt -Nname ${inputpath}/nInd.txt -outfiles sample2boot -whichPops all -boot 100

$fastNGSadmix -likes ${inputpath}/sample3.beagle.gz -fname ${inputpath}/refPanel.txt -Nname ${inputpath}/nInd.txt -outfiles sample3boot -whichPops all -boot 100


**Questions**
 - Compare the result for this sample with the previous one. Which reference populations does each draw on?
 - If the individual's true population is not in the panel, what will fastNGSadmix report instead?

Notice that now the FIRST row of the .qopt files, are the estimated ancestry based on ALL sites, and that the subsequent rows, are the ones based on the bootstraps.

Now let us plot the results for sample3 in R:

In [ ]:

# Plot estimates
admix<-read.table("sample3boot.qopt",as.is=T,h=T)
b<-barplot(as.matrix(admix[1,]),main="sample3",ylab="Admixture proportion",col="red",ylim=c(0,1))

# Plot confidence intervals
## - first we take the 0.025 and 0.975 sample quantiles for constructing the confidence interval for out estimates
lower<-as.numeric(apply(admix,2,function(x) quantile(x[2:length(x)],probs=c(0.025))))
upper<-as.numeric(apply(admix,2,function(x) quantile(x[2:length(x)],probs=c(0.975))))

## - then we plot them
segments(b,lower,b,upper)
segments(b-0.2,lower,b+0.2,lower)
segments(b-0.2,upper,b+0.2,upper)


- Can you say with confidence what the ancestry of this sample is?

Let us also plot sample2 with 100 bootstraps:

In [ ]:

# Plot estimates
admix<-read.table("sample2boot.qopt",as.is=T,h=T)
b<-barplot(as.matrix(admix[1,]),main="sample2",ylab="Admixture proportion",col="red")

# Plot confidence intervals
## - first we take the 0.025 and 0.975 sample quantiles for constructing the confidence interval for out estimates
lower<-as.numeric(apply(admix,2,function(x) quantile(x[2:length(x)],probs=c(0.025))))
upper<-as.numeric(apply(admix,2,function(x) quantile(x[2:length(x)],probs=c(0.975))))

## - then we plot them
segments(b,lower,b,upper)
segments(b-0.2,lower,b+0.2,lower)
segments(b-0.2,upper,b+0.2,upper)
                        

### Quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/admixture/quiz/admix_quiz5.json")


# C. The same questions with called genotypes: ADMIXTURE

NGSadmix works on genotype likelihoods because our data are low depth. When the data are good enough
that genotypes can simply be called - a SNP chip, or high depth sequencing - the same admixture model
is usually fitted with the program ADMIXTURE, and the convergence and model fit questions are exactly
the same.

The bonus exercise walks through that on a dataset of blue wildebeest, where the consequences of a
local optimum are considerably more dramatic than what you saw here:

[NGS admixture bonus notebook](https://github.com/popgenDK/courses/blob/main/advBinf/exercises/advBinf_admixture_bonus.ipynb)

Download it and upload it to the notebook server. If you get some odd error messages on the left hand
side of the screen when you upload it then don't worry about it - pressing "Console" will usually
make them go away.